In [11]:
# === Imports (один раз в начале ноутбука) ===
import sys, torch
import sys, importlib
from snn_mnist_net import Cfg, run_experiment, save_snn, load_weights_into, build_net,build_encoder_from_cfg
from label_map import build_label_map,save_label_map, load_label_map
from counts_readout import collect_counts_plus_cuda, make_mnist_datasets
from readout_models import tfidf_from_counts, train_mlp_readout
# (опционально) from evaluation import evaluate_on_mnist

_MODS = [
    "snn_mnist_net",
    "label_map",
    "counts_readout",
    "readout_models",
]

# 1) вычистим из sys.modules сам модуль и подпакеты
for m in list(sys.modules):
    if m in _MODS or any(m.startswith(x + ".") for x in _MODS):
        sys.modules.pop(m, None)

# 2) инвалидируем кэши файловой системы
importlib.invalidate_caches()

# 3) заново импортируем модули как модули
import snn_mnist_net, label_map, counts_readout, readout_models

In [12]:

# LATENCY config
# === FULL RUN CONFIG (адаптировано под Cfg) ===
cfg = Cfg(
    # базовые
    time=200, n_hidden=100, device="cpu", seed=42,
    # обучение
    N=60000,                 # стартовый «полный» прогон
    log_every=500,
    # STDP
    nu_plus=1e-4, nu_minus=-1e-3,
    # ингибиция / WTA
    enable_inhibition_at_start=True,
    top_k=3,
    inhib_strength=0.705,
    # веса
    w_init_lo=0.25, w_init_hi=0.80,
    w_clip_min=0.0, w_clip_max=1.0,
    # пороги/динамика LIF
    thresh_init=0.38, tau_val=150.0, refrac_val=2.0, reset_val=0.0,
    # кодировщик
    encoder="latency", poisson_rate_scale=0.011,
    # прочее
    debug=True,
    warmup_N = 100,
    thresh_min = 0.05,
    bootstrap_threshold = 0.12,
    rest_val = 0.0,
    latency_x_min = 0.05,
    encoder_rate_boost = 7
    
)
print("FULL RUN:", {k: getattr(cfg, k) for k in [
    "inhib_strength","w_init_lo","w_init_hi","poisson_rate_scale","thresh_init","N"
]})




FULL RUN: {'inhib_strength': 0.705, 'w_init_lo': 0.25, 'w_init_hi': 0.8, 'poisson_rate_scale': 0.011, 'thresh_init': 0.38, 'N': 60000}


In [13]:
import importlib, snn_mnist_net
importlib.reload(snn_mnist_net)
# === 1) Обучение SNN (STDP) ===
summary, FF, lif_layer, net, encoder = run_experiment(cfg, verbose=True, progress=True)
print("SUMMARY:", summary)

print("SUMMARY:", summary)
save_snn("out/snn_mnist_v6_latency.pt", cfg, FF, lif_layer)

Используем: cpu

SNN-MNIST TRAIN START
seed=42  device=cpu  T=200  N=60000
n_hidden=100  thresh_init=0.38
STDP: nu_plus=0.0001  nu_minus=-0.001
FF init: w∈[0.25, 0.8], clip=[0.0,1.0]
LIF: rest_val=0.0  reset_val=0.0  tau=150.0  refrac=2.0
Encoder=latency  poisson_rate_scale=0.011  base_seed=123
Inhibition: enable=True  inhib_strength=0.705  top_k=3
---------------------------

[lif] thresh = tensor mean=0.380 std=0.000 shape=(100,)
[lif] tc_decay = scalar 150.000
[lif] refrac = scalar 2.000
[lif] reset = scalar 0.000
[lif] rest  = scalar 0.000
[lif] dt = scalar 1.000


Warmup (T=200):   1%|▏                  | 1/100 [00:00<00:13,  7.12it/s, avg_rate=0.220 target=1.50]

[enc] shape=(200, 1, 784) sum=156.0 max=1.0
[enc] per_t: min=0.0 max=54.0 t_argmax=1
[mon] in_sum=156.0 lif_sum=4400.0 v_max=0.0 vt_min=0.11999999731779099
[W] mean=0.525335 max=0.799988 min=0.250008
[enc] shape=(200, 1, 784) sum=169.0 max=1.0
[enc] per_t: min=0.0 max=41.0 t_argmax=2


Warmup (T=200):   3%|▌                  | 3/100 [00:00<00:13,  7.31it/s, avg_rate=0.175 target=1.50]

[mon] in_sum=169.0 lif_sum=4000.0 v_max=0.0 vt_min=0.12424999475479126
[W] mean=0.525335 max=0.799988 min=0.250008
[enc] shape=(200, 1, 784) sum=118.0 max=1.0
[enc] per_t: min=0.0 max=20.0 t_argmax=0
[mon] in_sum=118.0 lif_sum=3500.0 v_max=0.0 vt_min=0.12845999002456665
[W] mean=0.525335 max=0.799988 min=0.250008


Warmup (T=200):  58%|██████████▍       | 58/100 [00:07<00:05,  7.52it/s, avg_rate=0.096 target=1.50]


KeyboardInterrupt: 

In [ ]:
# check latency network

In [6]:
import os
import importlib
import evaluation 
from evaluation import probe_readouts_counts
importlib.invalidate_caches()
importlib.reload(snn_mnist_net)
importlib.reload(evaluation)
importlib.invalidate_caches()
net, input_layer, lif_layer, connection, recurrent_inh, _ = build_net(cfg)
enc = build_encoder_from_cfg(cfg)

# 2) загрузить веса и пороги
load_weights_into(net, connection, lif_layer, "out/snn_mnist_v6_latency.pt")


[lif] thresh = tensor mean=0.380 std=0.000 shape=(100,)
[lif] tc_decay = scalar 150.000
[lif] refrac = scalar 2.000
[lif] reset = scalar 0.000
[lif] rest  = scalar 0.000
[lif] dt = scalar 1.000
Loaded from out/snn_mnist_v6_latency.pt


In [4]:
label_map_build = build_label_map(net, None, lif_layer, enc, n_calib=2000, T=cfg.time, top_k=3, seed=123)

save_label_map("out/label_v6_latency.pt",label_map_build, meta={"ckpt":"out/snn_mnist_v6_latency.pt","T":cfg.time})

Label-map built: 100/100 neurons assigned; active winners 100
[label_map] saved: out/label_v6_latency.pt  shape=(100,)


In [7]:
label_map_build = load_label_map("out/label_v6_latency.pt")

[label_map] loaded: out/label_v6_latency.pt  shape=(100,)  meta={'created_at': '2026-03-11 18:15:19', 'ckpt': 'out/snn_mnist_v6_latency.pt', 'T': 200}


In [8]:
from counts_readout import make_mnist_datasets

ds_train, ds_test = make_mnist_datasets()

In [14]:
from evaluation import eval_readouts_from_net
enc = build_encoder_from_cfg(cfg)
accs = eval_readouts_from_net(net, lif_layer, enc, cfg, label_map=label_map_build)
print(accs)   # ждём, что хотя бы counts_zscore+Linear или PCAwhiten+Linear > 0.6

[collect_counts_plus_cuda] offset summary: requested=+0.000, applied=0, avgΔvt=0.000, restore_ok=–


[collect_counts_plus_cuda] offset summary: requested=+0.000, applied=0, avgΔvt=0.000, restore_ok=–
{'counts_zscore+Linear': 0.21809999644756317, 'PCAwhiten+Linear': 0.21619999408721924, 'TFIDF+MLP': 0.2718000113964081}


In [10]:
print(accs)

{'counts_zscore+Linear': 0.21969999372959137, 'PCAwhiten+Linear': 0.21610000729560852, 'TFIDF+MLP': 0.27000001072883606}


# Test Poisson

In [17]:
import importlib, snn_mnist_net
importlib.reload(snn_mnist_net)
# === 1) Обучение SNN (STDP) ===
cfg.encoder = "poisson"
cfg.poisson_deterministic = True
summary, FF, lif_layer, net, encoder = run_experiment(cfg, verbose=True, progress=True)
print("SUMMARY:", summary)

print("SUMMARY:", summary)
save_snn("out/snn_mnist_v6_poisson_det.pt", cfg, FF, lif_layer)

Используем: cpu

SNN-MNIST TRAIN START
seed=42  device=cpu  T=200  N=60000
n_hidden=100  thresh_init=0.38
STDP: nu_plus=0.0001  nu_minus=-0.001
FF init: w∈[0.25, 0.8], clip=[0.0,1.0]
LIF: rest_val=0.0  reset_val=0.0  tau=150.0  refrac=2.0
Encoder=poisson  poisson_rate_scale=0.011  base_seed=123
Inhibition: enable=True  inhib_strength=0.705  top_k=3
---------------------------

[lif] thresh = tensor mean=0.380 std=0.000 shape=(100,)
[lif] tc_decay = scalar 150.000
[lif] refrac = scalar 2.000
[lif] reset = scalar 0.000
[lif] rest  = scalar 0.000
[lif] dt = scalar 1.000


Warmup (T=200):   1%|▏                  | 1/100 [00:00<00:14,  6.66it/s, avg_rate=0.290 target=1.50]

[enc] shape=(200, 1, 784) sum=249.0 max=1.0
[enc] per_t: min=0.0 max=6.0 t_argmax=130
[mon] in_sum=249.0 lif_sum=5800.0 v_max=0.0 vt_min=0.11999999731779099
[W] mean=0.525335 max=0.799988 min=0.250008
[enc] shape=(200, 1, 784) sum=274.0 max=1.0
[enc] per_t: min=0.0 max=4.0 t_argmax=14


Warmup (T=200):   3%|▌                  | 3/100 [00:00<00:13,  7.11it/s, avg_rate=0.280 target=1.50]

[mon] in_sum=274.0 lif_sum=6000.0 v_max=0.0 vt_min=0.12565000355243683
[W] mean=0.525335 max=0.799988 min=0.250008
[enc] shape=(200, 1, 784) sum=167.0 max=1.0
[enc] per_t: min=0.0 max=4.0 t_argmax=150
[mon] in_sum=167.0 lif_sum=5600.0 v_max=0.0 vt_min=0.13131999969482422
[W] mean=0.525335 max=0.799988 min=0.250008


...

[enc] shape=(200, 1, 784) sum=249.0 max=1.0
[enc] per_t: min=0.0 max=6.0 t_argmax=130
[W] mean=0.525162 max=0.799988 min=0.242163
[enc] shape=(200, 1, 784) sum=274.0 max=1.0
[enc] per_t: min=0.0 max=4.0 t_argmax=14


Train (N=60000, T=200):   0%| | 3/60000 [00:00<2:49:37,  5.90it/s, spikes=600.00 uniq=9 HHI=0.111 e≈

[W] mean=0.525027 max=0.799988 min=0.235773
[enc] shape=(200, 1, 784) sum=167.0 max=1.0
[enc] per_t: min=0.0 max=4.0 t_argmax=150
[W] mean=0.524968 max=0.799988 min=0.235773


Train (N=60000, T=200): 100%|█| 60000/60000 [2:37:32<00:00,  6.35it/s, spikes=450.36 uniq=100 HHI=0.

SUMMARY: {'time': 200, 'n_hidden': 100, 'N': 60000, 'seed': 42, 'device': 'cpu', 'log_every': 500, 'debug': True, 'nu_plus': 0.0001, 'nu_minus': -0.001, 'enable_inhibition_at_start': True, 'inhib_strength': 0.705, 'encoder': 'poisson', 'poisson_rate_scale': 0.011, 'poisson_base_seed': 123, 'thresh_init': 0.38, 'thresh_min': 0.05, 'thresh_max': 1.2, 'target_spikes': 1.5, 'ema_alpha': 0.9, 'ema_k': 0.02, 'tau_val': 150.0, 'refrac_val': 2.0, 'reset_val': 0.0, 'rest_val': 0.0, 'w_init_lo': 0.25, 'w_init_hi': 0.8, 'w_clip_min': 0.0, 'w_clip_max': 1.0, 'wmin': 0.0, 'wmax': 1.0, 'warmup_N': 100, 'top_k': 3, 'latency_x_min': 0.05, 'strong_inh_matrix': True, 'strong_inh_value': 0.65, 'bootstrap_threshold_enable': True, 'bootstrap_threshold': 0.12, 'bootstrap_refrac': 2.0, 'encoder_out_format': 'auto', 'poisson_deterministic': True, 'encoder_rate_boost': 7, 'spikes_per_sample': 450.36, 'synops_per_sample': 22426.79, 'v_updates_per_sample': 20000.0, 'energy_proxy_per_sample': 1671.6995, 'winners_

In [18]:
print("SUMMARY:", summary)

SUMMARY: {'time': 200, 'n_hidden': 100, 'N': 60000, 'seed': 42, 'device': 'cpu', 'log_every': 500, 'debug': True, 'nu_plus': 0.0001, 'nu_minus': -0.001, 'enable_inhibition_at_start': True, 'inhib_strength': 0.705, 'encoder': 'poisson', 'poisson_rate_scale': 0.011, 'poisson_base_seed': 123, 'thresh_init': 0.38, 'thresh_min': 0.05, 'thresh_max': 1.2, 'target_spikes': 1.5, 'ema_alpha': 0.9, 'ema_k': 0.02, 'tau_val': 150.0, 'refrac_val': 2.0, 'reset_val': 0.0, 'rest_val': 0.0, 'w_init_lo': 0.25, 'w_init_hi': 0.8, 'w_clip_min': 0.0, 'w_clip_max': 1.0, 'wmin': 0.0, 'wmax': 1.0, 'warmup_N': 100, 'top_k': 3, 'latency_x_min': 0.05, 'strong_inh_matrix': True, 'strong_inh_value': 0.65, 'bootstrap_threshold_enable': True, 'bootstrap_threshold': 0.12, 'bootstrap_refrac': 2.0, 'encoder_out_format': 'auto', 'poisson_deterministic': True, 'encoder_rate_boost': 7, 'spikes_per_sample': 450.36, 'synops_per_sample': 22426.79, 'v_updates_per_sample': 20000.0, 'energy_proxy_per_sample': 1671.6995, 'winners_